# Manipulation de fichiers CSV et JSON en Python — CORRIGÉ

**Public :** formation professionnelle (bases de Python acquises : variables, boucles, fonctions, listes et dictionnaires)

**Progression :**
| Niveau | Thème |
|---|---|
| 1 | Lire / écrire un CSV avec `csv.reader` / `csv.writer` |
| 2 | `DictReader` / `DictWriter`, agrégations, séparateurs |
| 3 | Bases du JSON : `load`, `dump`, `loads`, `dumps` |
| 4 | JSON imbriqué : parcourir, filtrer, modifier |
| 5 | Conversions CSV ⇄ JSON (aplatir / regrouper) |
| 6 | Robustesse : erreurs, données sales, validation, sérialisation |
| 7 | pandas : `read_csv`, `json_normalize`, `groupby`, `merge` |
| 8 | Mini-projet : pipeline de reporting |

**Avant de commencer :** exécuter la cellule de préparation ci-dessous. Elle crée un dossier `data/` avec les fichiers utilisés dans les exercices. On peut la ré-exécuter à tout moment pour repartir de fichiers propres.

In [ ]:
# Cellule de préparation : crée les fichiers de données utilisés dans les exercices.
import os, json

os.makedirs("data", exist_ok=True)

with open("data/ventes.csv", "w", encoding="utf-8", newline="") as f:
    f.write("""date,produit,categorie,quantite,prix_unitaire,ville
2026-01-05,Clavier,Informatique,3,45.0,Tunis
2026-01-05,Souris,Informatique,10,12.5,Sfax
2026-01-06,Chaise,Mobilier,2,120.0,Tunis
2026-01-07,Écran,Informatique,4,310.0,Sousse
2026-01-08,Bureau,Mobilier,1,450.0,Bizerte
2026-01-08,Clavier,Informatique,5,45.0,Sfax
2026-01-09,Lampe,Mobilier,6,35.0,Tunis
2026-01-10,Souris,Informatique,8,12.5,Bizerte
2026-01-11,Écran,Informatique,2,310.0,Tunis
2026-01-12,Chaise,Mobilier,4,120.0,Sousse
""")

# Fichier « sale » : séparateur ';', décimales avec virgule, lignes incomplètes ou invalides
with open("data/employes.csv", "w", encoding="utf-8", newline="") as f:
    f.write("""matricule;nom;service;salaire;date_embauche
E001;Martin Sophie;Informatique;2850,50;2019-03-15
E002;Bernard Lucas;RH;2100;2021-09-01
E003;Dubois Camille;Informatique;;2022-01-10
E004;Moreau Julien;Finance;3200,00;2018-06-20
E005;Laurent Chloé;RH;abc;2023-02-01
E006;Lefebvre Thomas;Informatique;3100;2017-11-05
E007;Girard Manon;Finance;2700;
""")

clients = [
    {"id": 1, "nom": "Dupont & Fils", "email": "contact@dupont-fils.fr",
     "adresse": {"ville": "Tunis", "code_postal": "1002"},
     "tags": ["premium", "b2b"], "actif": True},
    {"id": 2, "nom": "Boulangerie Lemoine", "email": "info@boulangerie-lemoine.fr",
     "adresse": {"ville": "Sfax", "code_postal": "3000"},
     "tags": ["b2b"], "actif": True},
    {"id": 3, "nom": "Garage Rousseau", "email": None,
     "adresse": {"ville": "Sousse", "code_postal": "4000"},
     "tags": [], "actif": False},
    {"id": 4, "nom": "Fontaine Informatique", "email": "bonjour@fontaine-info.fr",
     "adresse": {"ville": "Tunis", "code_postal": "1053"},
     "tags": ["premium"], "actif": True},
]
with open("data/clients.json", "w", encoding="utf-8") as f:
    json.dump(clients, f, ensure_ascii=False, indent=2)

commandes = {
    "magasin": "TechStore",
    "commandes": [
        {"numero": "C100", "client_id": 1, "date": "2026-02-01",
         "lignes": [{"produit": "Écran", "qte": 2, "prix": 310.0},
                    {"produit": "Clavier", "qte": 2, "prix": 45.0}]},
        {"numero": "C101", "client_id": 2, "date": "2026-02-03",
         "lignes": [{"produit": "Souris", "qte": 20, "prix": 12.5}]},
        {"numero": "C102", "client_id": 1, "date": "2026-02-10",
         "lignes": [{"produit": "Bureau", "qte": 1, "prix": 450.0},
                    {"produit": "Chaise", "qte": 4, "prix": 120.0},
                    {"produit": "Lampe", "qte": 2, "prix": 35.0}]},
        {"numero": "C103", "client_id": 4, "date": "2026-02-15",
         "lignes": [{"produit": "Écran", "qte": 3, "prix": 310.0}]},
    ],
}
with open("data/commandes.json", "w", encoding="utf-8") as f:
    json.dump(commandes, f, ensure_ascii=False, indent=2)

# Fichier JSON volontairement invalide (virgule finale)
with open("data/config_cassee.json", "w", encoding="utf-8") as f:
    f.write('{"app": "facturation", "version": 2, "debug": true,}')

print("Fichiers créés :", sorted(os.listdir("data")))

---
## Niveau 1 — Lire et écrire un CSV avec le module `csv`

**Rappels :**
```python
import csv
with open("fichier.csv", encoding="utf-8", newline="") as f:
    lecteur = csv.reader(f)          # chaque ligne -> liste de chaînes
    entete = next(lecteur)
    for ligne in lecteur: ...
```
- Toujours préciser `encoding="utf-8"` et `newline=""` (recommandé par la documentation).
- Toutes les valeurs lues sont des **chaînes** : penser à convertir (`int`, `float`).

**Exercice 1.1** — Ouvrir `data/ventes.csv` avec `csv.reader`, afficher l'en-tête puis chaque ligne.

In [ ]:
import csv

with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    lecteur = csv.reader(f)
    entete = next(lecteur)
    print("En-tête :", entete)
    for ligne in lecteur:
        print(ligne)

**Exercice 1.2** — Compter le nombre de ventes (lignes de données, sans l'en-tête) et le nombre de colonnes.

In [ ]:
import csv

with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    lignes = list(csv.reader(f))

entete, donnees = lignes[0], lignes[1:]
print("Colonnes :", len(entete))
print("Ventes   :", len(donnees))

**Exercice 1.3** — Calculer la quantité totale vendue (colonne `quantite`). Attention au type des valeurs lues.

In [ ]:
import csv

total = 0
with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    lecteur = csv.reader(f)
    entete = next(lecteur)
    i_qte = entete.index("quantite")
    for ligne in lecteur:
        total += int(ligne[i_qte])
print("Quantité totale :", total)

**Exercice 1.4** — Écrire avec `csv.writer` un fichier `data/produits.csv` contenant l'en-tête `produit,prix` et les trois lignes ci-dessous.

In [ ]:
import csv

produits = [("Clavier", 45.0), ("Souris", 12.5), ("Écran", 310.0)]

with open("data/produits.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["produit", "prix"])
    w.writerows(produits)

print(open("data/produits.csv", encoding="utf-8").read())

---
## Niveau 2 — `DictReader` et `DictWriter`

**Rappels :**
- `csv.DictReader(f)` : chaque ligne devient un **dictionnaire** `{nom_colonne: valeur}` — plus lisible et robuste à l'ordre des colonnes.
- `csv.DictWriter(f, fieldnames=[...])` puis `writeheader()` et `writerow(dict)`.
- Paramètre `delimiter=";"` pour les fichiers séparés par des points-virgules.

**Exercice 2.1** — Lire `data/ventes.csv` avec `DictReader` et afficher, pour chaque vente : `date - produit (ville)`.

In [ ]:
import csv

with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    for v in csv.DictReader(f):
        print(f"{v['date']} - {v['produit']} ({v['ville']})")

**Exercice 2.2** — Calculer le chiffre d'affaires total (`quantite × prix_unitaire`) et le chiffre d'affaires de la ville de Tunis.

In [ ]:
import csv

ca_total = 0.0
ca_tunis = 0.0
with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    for v in csv.DictReader(f):
        montant = int(v["quantite"]) * float(v["prix_unitaire"])
        ca_total += montant
        if v["ville"] == "Tunis":
            ca_tunis += montant
print(f"CA total : {ca_total:.2f} DT")
print(f"CA Tunis : {ca_tunis:.2f} DT")

**Exercice 2.3** — Construire un dictionnaire `ca_par_categorie` (catégorie → CA), puis l'afficher trié par CA décroissant.

In [ ]:
import csv

ca_par_categorie = {}
with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    for v in csv.DictReader(f):
        m = int(v["quantite"]) * float(v["prix_unitaire"])
        ca_par_categorie[v["categorie"]] = ca_par_categorie.get(v["categorie"], 0) + m

for cat, ca in sorted(ca_par_categorie.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat:<15}{ca:>10.2f}")

**Exercice 2.4** — Écrire dans `data/ventes_enrichies.csv` toutes les ventes avec une colonne supplémentaire `montant`. Utiliser `DictWriter`.

In [ ]:
import csv

with open("data/ventes.csv", encoding="utf-8", newline="") as fin, \
     open("data/ventes_enrichies.csv", "w", encoding="utf-8", newline="") as fout:
    lecteur = csv.DictReader(fin)
    colonnes = lecteur.fieldnames + ["montant"]
    w = csv.DictWriter(fout, fieldnames=colonnes)
    w.writeheader()
    for v in lecteur:
        v["montant"] = round(int(v["quantite"]) * float(v["prix_unitaire"]), 2)
        w.writerow(v)

print(open("data/ventes_enrichies.csv", encoding="utf-8").read())

**Exercice 2.5** — Le fichier `data/employes.csv` utilise `;` comme séparateur. Le lire et afficher la liste des services **distincts**.

In [ ]:
import csv

with open("data/employes.csv", encoding="utf-8", newline="") as f:
    services = {e["service"] for e in csv.DictReader(f, delimiter=";")}
print(sorted(services))

---
## Niveau 3 — Bases du JSON

**Rappels :**
| Python | JSON |
|---|---|
| `dict` | objet `{}` |
| `list`, `tuple` | tableau `[]` |
| `str`, `int`, `float` | chaîne, nombre |
| `True` / `False` / `None` | `true` / `false` / `null` |

- Fichier : `json.load(f)` / `json.dump(obj, f, indent=2, ensure_ascii=False)`
- Chaîne : `json.loads(texte)` / `json.dumps(obj)`

**Exercice 3.1** — Convertir le dictionnaire `produit` en chaîne JSON (avec `json.dumps`), afficher son type, puis la reconvertir en dictionnaire.

In [ ]:
import json

produit = {"nom": "Écran", "prix": 310.0, "en_stock": True, "remise": None}

texte = json.dumps(produit, ensure_ascii=False)
print(texte, type(texte))

retour = json.loads(texte)
print(retour, type(retour))
print(retour == produit)

**Exercice 3.2** — Comparer l'affichage de `json.dumps(produit)` avec et sans `indent=2` et `ensure_ascii=False`. Que change chaque option ?

In [ ]:
import json

produit = {"nom": "Écran", "prix": 310.0, "en_stock": True}
print(json.dumps(produit))
print(json.dumps(produit, indent=2, ensure_ascii=False))
# indent : met en forme sur plusieurs lignes (lisible par un humain)
# ensure_ascii=False : garde les accents au lieu de les échapper (\u00c9)

**Exercice 3.3** — Charger `data/clients.json`. Afficher le nombre de clients et le nom de chacun.

In [ ]:
import json

with open("data/clients.json", encoding="utf-8") as f:
    clients = json.load(f)

print(len(clients), "clients")
for c in clients:
    print("-", c["nom"])

**Exercice 3.4** — Sauvegarder dans `data/parametres.json` le dictionnaire de configuration ci-dessous, puis le relire pour vérifier.

In [ ]:
import json

parametres = {"app": "facturation", "version": 3, "devise": "DT", "tva": 0.19, "modules": ["ventes", "stock"]}

with open("data/parametres.json", "w", encoding="utf-8") as f:
    json.dump(parametres, f, indent=2, ensure_ascii=False)

with open("data/parametres.json", encoding="utf-8") as f:
    relu = json.load(f)
print(relu == parametres, relu)

---
## Niveau 4 — JSON imbriqué : parcourir, filtrer, modifier

**Rappels :**
- On accède aux niveaux successifs : `data["commandes"][0]["lignes"][1]["prix"]`
- `dict.get(cle, defaut)` évite une `KeyError` si une clé peut manquer.
- Une valeur `null` en JSON devient `None` en Python.

**Exercice 4.1** — Dans `data/clients.json`, afficher la ville de chaque client (`adresse → ville`), puis la liste des clients **actifs** de Tunis.

In [ ]:
import json

with open("data/clients.json", encoding="utf-8") as f:
    clients = json.load(f)

for c in clients:
    print(c["nom"], "->", c["adresse"]["ville"])

tunis = [c["nom"] for c in clients if c["actif"] and c["adresse"]["ville"] == "Tunis"]
print("Actifs à Tunis :", tunis)

**Exercice 4.2** — Lister les clients qui ont le tag `"premium"` et ceux qui n'ont **pas d'email** (`null`).

In [ ]:
import json

with open("data/clients.json", encoding="utf-8") as f:
    clients = json.load(f)

print("Premium   :", [c["nom"] for c in clients if "premium" in c["tags"]])
print("Sans email:", [c["nom"] for c in clients if c["email"] is None])

**Exercice 4.3** — Dans `data/commandes.json`, calculer le montant total de chaque commande (somme de `qte × prix` de ses lignes) et afficher `numero : montant`.

In [ ]:
import json

with open("data/commandes.json", encoding="utf-8") as f:
    data = json.load(f)

for cmd in data["commandes"]:
    montant = sum(l["qte"] * l["prix"] for l in cmd["lignes"])
    print(f"{cmd['numero']} : {montant:.2f} DT")

**Exercice 4.4** — Calculer la quantité totale vendue **par produit** sur toutes les commandes (dictionnaire produit → quantité).

In [ ]:
import json

with open("data/commandes.json", encoding="utf-8") as f:
    data = json.load(f)

qte_par_produit = {}
for cmd in data["commandes"]:
    for l in cmd["lignes"]:
        qte_par_produit[l["produit"]] = qte_par_produit.get(l["produit"], 0) + l["qte"]
print(qte_par_produit)

**Exercice 4.5** — Ajouter à chaque client un champ `nb_commandes` (calculé depuis `commandes.json`) et désactiver (`actif = False`) les clients sans commande. Sauvegarder dans `data/clients_maj.json`.

In [ ]:
import json

with open("data/clients.json", encoding="utf-8") as f:
    clients = json.load(f)
with open("data/commandes.json", encoding="utf-8") as f:
    commandes = json.load(f)["commandes"]

for c in clients:
    c["nb_commandes"] = sum(1 for cmd in commandes if cmd["client_id"] == c["id"])
    if c["nb_commandes"] == 0:
        c["actif"] = False

with open("data/clients_maj.json", "w", encoding="utf-8") as f:
    json.dump(clients, f, indent=2, ensure_ascii=False)

for c in clients:
    print(c["nom"], c["nb_commandes"], c["actif"])

---
## Niveau 5 — Conversions CSV ⇄ JSON

**Objectif :** passer d'un format tabulaire (CSV, « plat ») à un format hiérarchique (JSON, « imbriqué ») et inversement.
- CSV → JSON : chaque ligne `DictReader` devient un objet ; penser à convertir les types.
- JSON → CSV : il faut **aplatir** les structures imbriquées (ex. `adresse.ville` → colonne `ville`).

**Exercice 5.1** — Convertir `data/ventes.csv` en `data/ventes.json` (liste d'objets), avec `quantite` en entier et `prix_unitaire` en réel.

In [ ]:
import csv, json

with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    ventes = []
    for v in csv.DictReader(f):
        v["quantite"] = int(v["quantite"])
        v["prix_unitaire"] = float(v["prix_unitaire"])
        ventes.append(v)

with open("data/ventes.json", "w", encoding="utf-8") as f:
    json.dump(ventes, f, indent=2, ensure_ascii=False)

print(json.dumps(ventes[:2], indent=2, ensure_ascii=False))

**Exercice 5.2** — Convertir `data/clients.json` en `data/clients.csv` avec les colonnes `id, nom, email, ville, code_postal, tags, actif`. Les tags seront joints par `|` (ex. `premium|b2b`) et un email `None` deviendra une chaîne vide.

In [ ]:
import csv, json

with open("data/clients.json", encoding="utf-8") as f:
    clients = json.load(f)

colonnes = ["id", "nom", "email", "ville", "code_postal", "tags", "actif"]
with open("data/clients.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=colonnes)
    w.writeheader()
    for c in clients:
        w.writerow({
            "id": c["id"],
            "nom": c["nom"],
            "email": c["email"] or "",
            "ville": c["adresse"]["ville"],
            "code_postal": c["adresse"]["code_postal"],
            "tags": "|".join(c["tags"]),
            "actif": c["actif"],
        })

print(open("data/clients.csv", encoding="utf-8").read())

**Exercice 5.3** — « Aplatir » `data/commandes.json` en `data/lignes_commandes.csv` : **une ligne CSV par ligne de commande**, colonnes `numero, client_id, date, produit, qte, prix, montant`.

In [ ]:
import csv, json

with open("data/commandes.json", encoding="utf-8") as f:
    data = json.load(f)

colonnes = ["numero", "client_id", "date", "produit", "qte", "prix", "montant"]
with open("data/lignes_commandes.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=colonnes)
    w.writeheader()
    for cmd in data["commandes"]:
        for l in cmd["lignes"]:
            w.writerow({"numero": cmd["numero"], "client_id": cmd["client_id"],
                        "date": cmd["date"], **l, "montant": l["qte"] * l["prix"]})

print(open("data/lignes_commandes.csv", encoding="utf-8").read())

**Exercice 5.4** — Opération inverse : à partir de `data/ventes.csv`, produire un JSON **regroupé par ville** :
```json
{"Tunis": {"nb_ventes": 4, "ca": 1205.0, "produits": ["Chaise", "Clavier", ...]}, ...}
```
(liste de produits triée et sans doublons).

In [ ]:
import csv, json

par_ville = {}
with open("data/ventes.csv", encoding="utf-8", newline="") as f:
    for v in csv.DictReader(f):
        d = par_ville.setdefault(v["ville"], {"nb_ventes": 0, "ca": 0.0, "produits": set()})
        d["nb_ventes"] += 1
        d["ca"] += int(v["quantite"]) * float(v["prix_unitaire"])
        d["produits"].add(v["produit"])

# un set n'est pas sérialisable en JSON -> conversion en liste triée
for d in par_ville.values():
    d["produits"] = sorted(d["produits"])

with open("data/ventes_par_ville.json", "w", encoding="utf-8") as f:
    json.dump(par_ville, f, indent=2, ensure_ascii=False)
print(json.dumps(par_ville, indent=2, ensure_ascii=False))

---
## Niveau 6 — Robustesse : erreurs, données sales, validation

**Rappels :**
- Fichier absent → `FileNotFoundError` ; JSON mal formé → `json.JSONDecodeError` (attributs `lineno`, `colno`, `msg`).
- Conversion impossible → `ValueError` (ex. `float("abc")`).
- Bonne pratique : **ne pas arrêter tout le traitement** pour une ligne invalide ; la journaliser et continuer.
- Les objets non natifs (`datetime`, `Decimal`, `set`…) ne sont pas sérialisables directement : utiliser le paramètre `default=` de `json.dump`.

**Exercice 6.1** — Écrire une fonction `charger_json(chemin)` qui renvoie le contenu du fichier, ou `None` en affichant un message clair si le fichier n'existe pas ou si le JSON est invalide (indiquer ligne et colonne). La tester sur `data/clients.json`, `data/inexistant.json` et `data/config_cassee.json`.

In [ ]:
import json

def charger_json(chemin):
    try:
        with open(chemin, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"[ERREUR] Fichier introuvable : {chemin}")
    except json.JSONDecodeError as e:
        print(f"[ERREUR] JSON invalide dans {chemin} (ligne {e.lineno}, colonne {e.colno}) : {e.msg}")
    return None

print(type(charger_json("data/clients.json")))
print(charger_json("data/inexistant.json"))
print(charger_json("data/config_cassee.json"))

**Exercice 6.2** — Écrire une fonction `vers_float(texte)` qui convertit une chaîne en réel en acceptant la virgule décimale (`"2850,50"` → `2850.5`) et renvoie `None` si la valeur est vide ou invalide.

In [ ]:
def vers_float(texte):
    if texte is None or texte.strip() == "":
        return None
    try:
        return float(texte.strip().replace(",", "."))
    except ValueError:
        return None

print(vers_float("2850,50"), vers_float("2100"), vers_float(""), vers_float("abc"))
# 2850.5 2100.0 None None

**Exercice 6.3** — Nettoyer `data/employes.csv` :
- les lignes valides (salaire convertible **et** date d'embauche présente) vont dans `data/employes_propres.csv` (séparateur `,`, salaire en réel avec un point) ;
- les lignes rejetées vont dans `data/employes_rejets.csv` avec une colonne `motif`.

Afficher le nombre de lignes valides et rejetées.

In [ ]:
import csv

valides, rejets = [], []
with open("data/employes.csv", encoding="utf-8", newline="") as f:
    lecteur = csv.DictReader(f, delimiter=";")
    colonnes = lecteur.fieldnames
    for e in lecteur:
        salaire = vers_float(e["salaire"])
        if salaire is None:
            rejets.append({**e, "motif": f"salaire invalide : '{e['salaire']}'"})
        elif not e["date_embauche"]:
            rejets.append({**e, "motif": "date d'embauche manquante"})
        else:
            e["salaire"] = salaire
            valides.append(e)

with open("data/employes_propres.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=colonnes)
    w.writeheader(); w.writerows(valides)

with open("data/employes_rejets.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=colonnes + ["motif"])
    w.writeheader(); w.writerows(rejets)

print(f"{len(valides)} valides, {len(rejets)} rejetées")
for r in rejets:
    print(" -", r["matricule"], ":", r["motif"])

**Exercice 6.4** — Le dictionnaire `rapport` contient une date (`datetime`) et un ensemble. `json.dumps(rapport)` échoue : le vérifier, puis écrire une fonction `convertir(obj)` à passer en `default=` pour sérialiser les dates au format ISO et les ensembles en listes triées.

In [ ]:
import json
from datetime import datetime

rapport = {"genere_le": datetime(2026, 3, 1, 9, 30), "villes": {"Tunis", "Sfax"}, "total": 4512.5}

try:
    json.dumps(rapport)
except TypeError as e:
    print("Erreur :", e)

def convertir(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(obj)
    raise TypeError(f"Type non sérialisable : {type(obj).__name__}")

print(json.dumps(rapport, default=convertir, indent=2))

**Exercice 6.5** — Écrire une fonction `valider_client(c)` qui renvoie la **liste des erreurs** d'un client : champs obligatoires `id` (int), `nom` (str non vide), `adresse.ville` (str) ; si `email` n'est pas `None`, il doit contenir `@`. Tester sur `data/clients.json` et sur le client invalide fourni.

In [ ]:
import json

client_invalide = {"id": "5", "nom": "", "email": "pas-un-email", "adresse": {}}

def valider_client(c):
    erreurs = []
    if not isinstance(c.get("id"), int):
        erreurs.append("id doit être un entier")
    if not isinstance(c.get("nom"), str) or not c["nom"].strip():
        erreurs.append("nom obligatoire")
    if not isinstance(c.get("adresse", {}).get("ville"), str):
        erreurs.append("adresse.ville obligatoire")
    email = c.get("email")
    if email is not None and "@" not in email:
        erreurs.append(f"email invalide : {email}")
    return erreurs

with open("data/clients.json", encoding="utf-8") as f:
    for c in json.load(f):
        print(c["nom"], "->", valider_client(c) or "OK")
print("client_invalide ->", valider_client(client_invalide))

---
## Niveau 7 — Aller plus vite avec pandas

**Rappels :**
```python
import pandas as pd
df = pd.read_csv("fichier.csv")             # sep=";", decimal=","...
df = pd.read_json("fichier.json")
df = pd.json_normalize(liste_de_dicts)      # aplatit le JSON imbriqué
df.to_csv("sortie.csv", index=False)
df.to_json("sortie.json", orient="records", force_ascii=False, indent=2)
```

**Exercice 7.1** — Charger `data/ventes.csv` dans un DataFrame. Afficher les 5 premières lignes, les types des colonnes (`dtypes`) et le nombre de lignes.

In [ ]:
import pandas as pd

df = pd.read_csv("data/ventes.csv")
print(df.head())
print(df.dtypes)
print(len(df), "lignes")

**Exercice 7.2** — Ajouter une colonne `montant`, puis calculer le CA par ville et par catégorie (`groupby`). Refaire en une ligne l'exercice 2.3.

In [ ]:
import pandas as pd

df = pd.read_csv("data/ventes.csv")
df["montant"] = df["quantite"] * df["prix_unitaire"]
print(df.groupby("ville")["montant"].sum().sort_values(ascending=False))
print(df.groupby("categorie")["montant"].sum().sort_values(ascending=False))

**Exercice 7.3** — Lire `data/employes.csv` avec les bons paramètres (`sep`, `decimal`). Les valeurs invalides du salaire doivent devenir `NaN` (`pd.to_numeric(..., errors="coerce")`). Afficher le salaire moyen par service et les employés dont une valeur est manquante.

In [ ]:
import pandas as pd

df = pd.read_csv("data/employes.csv", sep=";", decimal=",")
df["salaire"] = pd.to_numeric(df["salaire"], errors="coerce")
print(df)
print(df.groupby("service")["salaire"].mean().round(2))
print("Valeurs manquantes :")
print(df[df.isna().any(axis=1)][["matricule", "nom"]])

**Exercice 7.4** — Avec `pd.json_normalize`, aplatir `data/commandes.json` en un DataFrame d'une ligne par ligne de commande (paramètres `record_path` et `meta`). Calculer le montant total par client, puis l'exporter en JSON (`orient="records"`).

In [ ]:
import json
import pandas as pd

with open("data/commandes.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data["commandes"], record_path="lignes",
                       meta=["numero", "client_id", "date"])
df["montant"] = df["qte"] * df["prix"]
print(df)

totaux = df.groupby("client_id", as_index=False)["montant"].sum()
totaux.to_json("data/totaux_clients.json", orient="records", indent=2, force_ascii=False)
print(open("data/totaux_clients.json", encoding="utf-8").read())

**Exercice 7.5** — Jointure : aplatir `data/clients.json` (`json_normalize`), le fusionner (`merge`) avec les totaux de l'exercice précédent pour obtenir `nom, adresse.ville, montant` (0 pour les clients sans commande), puis exporter en CSV.

In [ ]:
import json
import pandas as pd

with open("data/clients.json", encoding="utf-8") as f:
    clients = pd.json_normalize(json.load(f))

totaux = pd.read_json("data/totaux_clients.json")
res = clients.merge(totaux, left_on="id", right_on="client_id", how="left")
res["montant"] = res["montant"].fillna(0)
res = res[["nom", "adresse.ville", "montant"]].sort_values("montant", ascending=False)
res.to_csv("data/synthese_clients.csv", index=False)
print(res)

---
## Niveau 8 — Mini-projet : pipeline de reporting

Ce dernier exercice reprend l'ensemble des notions. Il peut être réalisé **avec ou sans pandas**.

**Exercice 8.1 — Pipeline complet**

Écrire un script structuré en fonctions qui :
1. charge `data/commandes.json` et `data/clients.json` (avec gestion d'erreurs) ;
2. calcule pour chaque client : nombre de commandes, CA total, panier moyen, produit le plus acheté (en quantité) ;
3. exporte `data/rapport_clients.csv` (une ligne par client, trié par CA décroissant) ;
4. exporte `data/rapport.json` au format :
```json
{
  "magasin": "TechStore",
  "genere_le": "2026-...T...",
  "ca_global": 2890.0,
  "meilleur_client": "Dupont & Fils",
  "clients": [ {...}, ... ]
}
```
5. affiche un résumé lisible dans la console.

*Bonus : rendre les chemins de fichiers paramétrables (arguments de fonction) et ajouter quelques `assert` de vérification.*

In [ ]:
import csv, json
from datetime import datetime


def charger(chemin):
    try:
        with open(chemin, encoding="utf-8") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        raise SystemExit(f"Impossible de charger {chemin} : {e}")


def stats_clients(clients, commandes):
    resultats = []
    for c in clients:
        cmds = [cmd for cmd in commandes if cmd["client_id"] == c["id"]]
        ca = sum(l["qte"] * l["prix"] for cmd in cmds for l in cmd["lignes"])
        qtes = {}
        for cmd in cmds:
            for l in cmd["lignes"]:
                qtes[l["produit"]] = qtes.get(l["produit"], 0) + l["qte"]
        resultats.append({
            "id": c["id"],
            "nom": c["nom"],
            "ville": c["adresse"]["ville"],
            "nb_commandes": len(cmds),
            "ca": round(ca, 2),
            "panier_moyen": round(ca / len(cmds), 2) if cmds else 0.0,
            "produit_favori": max(qtes, key=qtes.get) if qtes else None,
        })
    return sorted(resultats, key=lambda r: r["ca"], reverse=True)


def exporter_csv(stats, chemin):
    with open(chemin, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(stats[0].keys()))
        w.writeheader()
        w.writerows(stats)


def exporter_json(magasin, stats, chemin):
    rapport = {
        "magasin": magasin,
        "genere_le": datetime.now().isoformat(timespec="seconds"),
        "ca_global": round(sum(s["ca"] for s in stats), 2),
        "meilleur_client": stats[0]["nom"],
        "clients": stats,
    }
    with open(chemin, "w", encoding="utf-8") as f:
        json.dump(rapport, f, indent=2, ensure_ascii=False)
    return rapport


def pipeline(f_commandes="data/commandes.json", f_clients="data/clients.json",
             f_csv="data/rapport_clients.csv", f_json="data/rapport.json"):
    data = charger(f_commandes)
    clients = charger(f_clients)
    stats = stats_clients(clients, data["commandes"])
    exporter_csv(stats, f_csv)
    rapport = exporter_json(data["magasin"], stats, f_json)

    # vérifications
    total_direct = sum(l["qte"] * l["prix"] for cmd in data["commandes"] for l in cmd["lignes"])
    assert abs(rapport["ca_global"] - total_direct) < 1e-6
    assert len(stats) == len(clients)

    print(f"Rapport {rapport['magasin']} — CA global : {rapport['ca_global']:.2f} DT")
    print(f"{'Client':<24}{'Cmd':>4}{'CA':>10}{'Panier':>10}  Favori")
    for s in stats:
        print(f"{s['nom']:<24}{s['nb_commandes']:>4}{s['ca']:>10.2f}{s['panier_moyen']:>10.2f}  {s['produit_favori'] or '-'}")


pipeline()